# LLO 9.1: Build Model on Various Data Sets
**Part A:** Credit Card Fraud Detection  
**Part B1:** Twitter Sentiment Analysis  
**Part B2:** HotpotQA Question Answering

## Google Colab instructions
1. Open this notebook in Google Colab.
2. Select **Runtime → Change runtime type → T4 GPU** (recommended for Part B2; CPU also works but is slower).
3. Run cells from top to bottom. Do not skip setup cells.
4. Part A uses the public Kaggle credit-card dataset. If automatic download fails, upload `creditcard.csv` when prompted.
5. Part B downloads datasets and a pre-trained QA model from Hugging Face, so an internet connection is required.

**Note:** Dataset/model downloads may take time. The notebook uses a smaller HotpotQA validation sample to keep the practical manageable.


## PART A: Credit Card Fraud Detection (Machine Learning)

### A1. Import libraries

In [ ]:
# Install dependencies (run this cell first in Colab)
%pip -q install -U numpy pandas matplotlib seaborn scikit-learn datasets transformers kagglehub

import os
import re
import string
import collections
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score,
    average_precision_score, roc_curve, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score, accuracy_score
)

warnings.filterwarnings("ignore")
print("Libraries imported successfully. If Colab asks for a runtime restart, restart and run cells from the top.")


### A2. Load the data set
The file is `creditcard.csv` (Kaggle: *Credit Card Fraud Detection* by ULB). Either upload it using the left Files panel in Colab, or let the cell below download it.

In [ ]:
# Load creditcard.csv. First try KaggleHub; if that fails, upload the CSV manually.
import os
import pandas as pd

CSV_PATH = "creditcard.csv"

if not os.path.exists(CSV_PATH):
    try:
        import kagglehub
        print("Trying to download the public dataset with KaggleHub...")
        downloaded_path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
        possible_path = os.path.join(downloaded_path, "creditcard.csv")
        if os.path.exists(possible_path):
            CSV_PATH = possible_path
        else:
            raise FileNotFoundError("KaggleHub download completed, but creditcard.csv was not found.")
    except Exception as err:
        print("Automatic download did not work:", err)
        print("Please upload creditcard.csv from the Kaggle Credit Card Fraud Detection dataset.")
        try:
            from google.colab import files
            uploaded = files.upload()
            if "creditcard.csv" not in uploaded:
                raise FileNotFoundError("Please upload a file named exactly creditcard.csv.")
            CSV_PATH = "creditcard.csv"
        except ImportError:
            raise FileNotFoundError(
                "creditcard.csv is missing. In Colab, upload it using the Files panel; "
                "locally, place it beside this notebook."
            )

df = pd.read_csv(CSV_PATH)
required_cols = {"Time", "Amount", "Class"}
missing_cols = required_cols - set(df.columns)
if missing_cols:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing_cols)}")
if df.empty:
    raise ValueError("creditcard.csv contains no rows.")

print("Shape:", df.shape)
display(df.head())


**SCREENSHOT 1:** the output above (shape and first 5 rows).

### A3. Explore the data

In [ ]:
df.info()
print("\\nMissing values:", int(df.isnull().sum().sum()))
display(df.describe().T.head(8))


**SCREENSHOT 2:** the output above (data types, missing values, summary).

In [ ]:
counts = df["Class"].value_counts().sort_index()
print(counts)
fraud_count = int((df["Class"] == 1).sum())
print("Fraud percentage: %.3f%%" % (100 * fraud_count / len(df)))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
sns.countplot(x="Class", data=df, ax=ax[0])
ax[0].set_title("Class distribution (0 = genuine, 1 = fraud)")
sns.boxplot(x="Class", y="Amount", data=df, ax=ax[1])
ax[1].set_yscale("symlog")
ax[1].set_title("Transaction amount by class (symlog scale)")
plt.tight_layout()
plt.show()


**SCREENSHOT 3:** class counts, fraud percentage and both charts. This shows the data is highly imbalanced.

### A4. Pre-processing and train/test split

In [ ]:
# Split first, then fit scalers only on training data to avoid data leakage.
X_raw = df.drop(columns=["Class"]).copy()
y = df["Class"].astype(int)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
for col in ["Amount", "Time"]:
    X_train_raw[col] = scaler.fit_transform(X_train_raw[[col]])
    X_test_raw[col] = scaler.transform(X_test_raw[[col]])

X_train = X_train_raw
X_test = X_test_raw

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Fraud in train: %d | Fraud in test: %d" % (int(y_train.sum()), int(y_test.sum())))


**SCREENSHOT 4:** the split sizes above.

### A5. Build the models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, class_weight="balanced_subsample",
        n_jobs=-1, random_state=42
    ),
}
probs, preds = {}, {}
for name, model in models.items():
    model.fit(X_train, y_train)
    probs[name] = model.predict_proba(X_test)[:, 1]
    preds[name] = model.predict(X_test)
    print(name, "trained")


### A6. Evaluate the models

In [ ]:
for name in models:
    print("=" * 55)
    print(name)
    print("=" * 55)
    print(classification_report(
        y_test, preds[name], labels=[0, 1],
        digits=4, target_names=["Genuine", "Fraud"], zero_division=0
    ))
    print("ROC-AUC:", round(roc_auc_score(y_test, probs[name]), 4))
    print("PR-AUC :", round(average_precision_score(y_test, probs[name]), 4), "\\n")


**SCREENSHOT 5:** classification reports with ROC-AUC and PR-AUC for both models.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, name in zip(ax, models):
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, preds[name], labels=[0, 1]),
        display_labels=["Genuine", "Fraud"]
    ).plot(ax=a, cmap="Greys", colorbar=False)
    a.set_title(name)
plt.tight_layout()
plt.show()


**SCREENSHOT 6:** both confusion matrices.

In [ ]:
plt.figure(figsize=(6, 5))
for name in models:
    fpr, tpr, _ = roc_curve(y_test, probs[name])
    plt.plot(fpr, tpr, label="%s (AUC = %.3f)" % (name, roc_auc_score(y_test, probs[name])))
plt.plot([0, 1], [0, 1], "k--", label="Random guess")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curve comparison")
plt.legend()
plt.show()

summary = pd.DataFrame({
    n: {
        "Precision": precision_score(y_test, preds[n], zero_division=0),
        "Recall": recall_score(y_test, preds[n], zero_division=0),
        "F1": f1_score(y_test, preds[n], zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, probs[n]),
        "PR-AUC": average_precision_score(y_test, probs[n])
    } for n in models
}).T.round(4)
display(summary)


**SCREENSHOT 7:** ROC curve and the final comparison table (fraud class).

## PART B1: Twitter Data Set (NLP, Sentiment Analysis)

Install the libraries first (needed once per Colab session).

In [ ]:
# Install NLP dependencies (safe to re-run in Colab)
%pip -q install -U datasets transformers
print("NLP dependencies ready.")


### B1.1 Load the Twitter data set
This uses the TweetEval sentiment data set (tweets labelled negative / neutral / positive), loaded directly from Hugging Face.

In [ ]:
from datasets import load_dataset

tw = load_dataset("cardiffnlp/tweet_eval", "sentiment")
label_names = {0: "negative", 1: "neutral", 2: "positive"}

train_df = tw["train"].to_pandas()
test_df = tw["test"].to_pandas()
train_df["sentiment"] = train_df["label"].map(label_names)
test_df["sentiment"] = test_df["label"].map(label_names)

print("Train:", train_df.shape, " Test:", test_df.shape)
display(train_df.head(8))


**SCREENSHOT 8:** data set size and sample tweets.

In [ ]:
print(train_df["sentiment"].value_counts())
train_df["sentiment"].value_counts().plot(kind="bar", color="gray", title="Tweet sentiment distribution (training set)")
plt.xticks(rotation=0); plt.ylabel("Number of tweets"); plt.show()

**SCREENSHOT 9:** label counts and bar chart.

### B1.2 Text pre-processing

In [ ]:
def clean(t):
    t = str(t).lower()
    t = re.sub(r"http\\S+|www\\.\\S+", " ", t)   # links
    t = re.sub(r"@\\w+", " user ", t)              # mentions
    t = re.sub(r"#", "", t)                          # hashtag symbol
    t = re.sub(r"[^a-z\\s']", " ", t)              # punctuation, digits, emojis
    return re.sub(r"\\s+", " ", t).strip()

train_df["clean"] = train_df["text"].apply(clean)
test_df["clean"] = test_df["text"].apply(clean)
for i in range(min(4, len(train_df))):
    print("ORIGINAL:", train_df.iloc[i]["text"])
    print("CLEANED :", train_df.iloc[i]["clean"])
    print()


**SCREENSHOT 10:** original vs cleaned tweets.

### B1.3 Feature extraction and model building

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
Xtr = tfidf.fit_transform(train_df["clean"])
ytr = train_df["label"].astype(int)
Xte = tfidf.transform(test_df["clean"])
yte = test_df["label"].astype(int)
print("TF-IDF matrix:", Xtr.shape)

nlp_models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000)
}
nlp_pred = {}
for name, model in nlp_models.items():
    model.fit(Xtr, ytr)
    nlp_pred[name] = model.predict(Xte)
    print("%-20s accuracy = %.4f" % (name, accuracy_score(yte, nlp_pred[name])))


**SCREENSHOT 11:** TF-IDF shape and accuracy of both models.

In [ ]:
best = max(nlp_pred, key=lambda name: accuracy_score(yte, nlp_pred[name]))
print("Best model:", best, "\\n")
print(classification_report(
    yte, nlp_pred[best], labels=[0, 1, 2],
    target_names=["negative", "neutral", "positive"], digits=4, zero_division=0
))
ConfusionMatrixDisplay(
    confusion_matrix(yte, nlp_pred[best], labels=[0, 1, 2]),
    display_labels=["negative", "neutral", "positive"]
).plot(cmap="Greys", colorbar=False)
plt.title("Confusion matrix: " + best)
plt.show()


**SCREENSHOT 12:** classification report and confusion matrix of the best model.

In [ ]:
samples = [
    "I absolutely love this new phone, the camera is amazing!",
    "Worst customer service ever, I want my money back.",
    "The meeting is scheduled for 3 pm tomorrow."
]
for sample in samples:
    prediction = int(nlp_models[best].predict(tfidf.transform([clean(sample)]))[0])
    print("%-12s <- %s" % (label_names[prediction].upper(), sample))


**SCREENSHOT 13:** predictions on new tweets (live demonstration of the model).

## PART B2: HotpotQA Data Set (NLP, Question Answering)

HotpotQA contains multi-hop questions that need information from more than one paragraph. Here a pre-trained question-answering model (DistilBERT fine-tuned on SQuAD) is applied to HotpotQA and evaluated with Exact Match (EM) and F1, which are the standard metrics for this task.

In [ ]:
# Download a small validation sample for a manageable practical run.
hq = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation[:100]")
print(hq)
ex = hq[0]
print("\\nQUESTION:", ex["question"])
print("ANSWER  :", ex["answer"])
print("TYPE    :", ex["type"])
print("LEVEL   :", ex["level"])
print("SUPPORTING TITLES:", ex["supporting_facts"]["title"])


**SCREENSHOT 14:** data set structure and one sample question with its answer.

In [ ]:
def build_context(example):
    # HotpotQA context is a list of (title, sentences) pairs.
    parts = []
    for title, sentences in zip(example["context"]["title"], example["context"]["sentences"]):
        parts.append(title + ". " + " ".join(sentences))
    return " ".join(parts)

import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

model_name = "distilbert-base-cased-distilled-squad"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
qa_model = AutoModelForQuestionAnswering.from_pretrained(model_name).to(device).eval()

def qa(question, context, max_answer_len=30):
    encoded = tok(
        question, context,
        truncation="only_second",
        max_length=384,
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding=True,
        return_tensors="pt"
    )
    offsets = encoded["offset_mapping"]
    model_inputs = {
        key: encoded[key].to(device)
        for key in ("input_ids", "attention_mask")
    }

    with torch.no_grad():
        output = qa_model(**model_inputs)

    best_score, best_text = float("-inf"), ""
    for i in range(model_inputs["input_ids"].shape[0]):
        sequence_ids = encoded.sequence_ids(i)
        offset_list = offsets[i].tolist()
        start_scores = output.start_logits[i].detach().cpu().numpy()
        end_scores = output.end_logits[i].detach().cpu().numpy()

        # Consider only tokens belonging to the context, not the question/special tokens.
        starts = np.argsort(start_scores)[-20:]
        ends = np.argsort(end_scores)[-20:]
        for start_idx in starts:
            for end_idx in ends:
                if sequence_ids[start_idx] != 1 or sequence_ids[end_idx] != 1:
                    continue
                if end_idx < start_idx or end_idx - start_idx + 1 > max_answer_len:
                    continue
                start_char = offset_list[start_idx][0]
                end_char = offset_list[end_idx][1]
                if end_char <= start_char:
                    continue
                score = float(start_scores[start_idx] + end_scores[end_idx])
                if score > best_score:
                    best_score = score
                    best_text = context[start_char:end_char]

    return {"answer": best_text.strip(), "score": best_score}

print("Question-answering model loaded successfully.")


In [ ]:
def norm(text):
    text = str(text).lower()
    text = "".join(char for char in text if char not in string.punctuation)
    text = re.sub(r"\\b(a|an|the)\\b", " ", text)
    return " ".join(text.split())

def f1(prediction, gold):
    pred_tokens = norm(prediction).split()
    gold_tokens = norm(gold).split()
    if not pred_tokens or not gold_tokens:
        return float(pred_tokens == gold_tokens)
    common = collections.Counter(pred_tokens) & collections.Counter(gold_tokens)
    same = sum(common.values())
    if same == 0:
        return 0.0
    precision = same / len(pred_tokens)
    recall = same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

rows = []
# Evaluate the same 100 examples loaded above; keep context size bounded for Colab.
for idx, example in enumerate(hq):
    context = build_context(example)[:6000]
    output = qa(example["question"], context)
    rows.append({
        "question": example["question"],
        "gold": example["answer"],
        "predicted": output["answer"],
        "EM": int(norm(output["answer"]) == norm(example["answer"])),
        "F1": f1(output["answer"], example["answer"]),
        "type": example["type"]
    })
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1}/{len(hq)} questions")

res = pd.DataFrame(rows)
print("Questions evaluated:", len(res))
print("Exact Match: %.2f%%" % (100 * res["EM"].mean()))
print("F1 score   : %.2f%%" % (100 * res["F1"].mean()))
display(res[["question", "gold", "predicted", "EM"]].head(8))


**SCREENSHOT 15:** EM and F1 scores with the first few predictions compared with the correct answers.

In [ ]:
display(res.groupby("type")[["EM", "F1"]].mean().round(3))
res.groupby("type")[["EM", "F1"]].mean().plot(
    kind="bar", color=["gray", "black"],
    title="HotpotQA performance by question type"
)
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.show()


**SCREENSHOT 16:** performance by question type (bridge vs comparison).

## Conclusion (write this in your own words in the journal)
- **Credit card fraud:** only about 0.17% of transactions are fraud, so accuracy is misleading; precision, recall, F1 and PR-AUC give the true picture.
- **Twitter sentiment:** TF-IDF features with a linear classifier give a solid baseline, and the neutral class is usually the hardest to separate.
- **HotpotQA:** a pre-trained model built for single-passage QA struggles on multi-hop questions, which shows why dedicated multi-hop methods are needed.